# Desproporcionalidade

In [19]:
import sys
import pandas as pd
 
sys.path.append('../../src/')
 
# Configurações de exibição
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Banco em arquivo (persiste durante a sessão).
#%sql duckdb:///content/vigimed.duckdb
# Se preferir memória: 
%sql duckdb:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


# 1. Iniciando a seção

In [2]:
%sql ROLLBACK;

Running query in 'duckdb:///:memory:'

,Success


# 2. Criando as tabelas

In [3]:
%%sql
DROP TABLE IF EXISTS dim_atc;
CREATE TABLE dim_atc AS
SELECT * FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_atc\dim_atc.parquet');

DROP TABLE IF EXISTS fat_medicamentos;
CREATE TABLE fat_medicamentos AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\fat_medicamentos\fat_medicamentos.parquet');

DROP TABLE IF EXISTS dim_notificacoes;
CREATE TABLE dim_notificacoes AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_notificacoes\dim_notificacoes.parquet');

DROP TABLE IF EXISTS fat_reacoes;
CREATE TABLE fat_reacoes AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\fat_reacoes\fat_reacoes.parquet');

DROP TABLE IF EXISTS dim_soc_llt;
CREATE TABLE dim_soc_llt AS
SELECT *
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_soc_llt\dim_soc_llt.parquet');
 

Running query in 'duckdb:///:memory:'

,Success


In [8]:
%%sql 
select * from dim_notificacoes limit 2

Running query in 'duckdb:///:memory:'

,id,UF,TIPO_ENTRADA_VIGIMED,RECEBIDO_DE,IDENTIFICACAO_NOTIFICACAO,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO,NOTIFICACAO_PARENT_CHILD,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO,GRUPO_IDADE,IDADE_GESTACIONAL_MOMENTO_REACAO,SEXO,GESTANTE,LACTANTE,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE,GRAVIDADE,DESFECHO,DATA_INICIO_HORA,DATA_FINAL_HORA,...,SEXO_CHAVE,GESTANTE_VALOR,GESTANTE_CHAVE,LACTANTE_VALOR,LACTANTE_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,RELACAO_MEDICAMENTO_EVENTO_CHAVE,ACAO_ADOTADA_VALOR,ACAO_ADOTADA_CHAVE,NOTIFICADOR_VALOR,NOTIFICADOR_CHAVE,IDADE_MOMENTO_REACAO_TIPO_CHAVE,IDADE_MOMENTO_REACAO_TIPO_VALOR,IDADE_MOMENTO_REACAO_VALOR,DURACAO_TIPO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR,HASH_SILVER
0,BR-ANVISA-300212656,SP,EMPRESAS FARMACEUTICAS,EMPRESA FARMACEUTICA,BR-ANVISA-300212656,2023-09-28,2023-09-28,NaT,NOTIFICACAO ESPONTANEA,None,1990-01-31,30 ano,None,None,Feminino,Năo,Năo,68.0,165,Hemiparesia,Sim,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,RECUPERADO,2021-01-25,2021-02-06,...,1,NĂO,2,NĂO,2,SIM,1,RECUPERADO,1,CONCOMITANTE,2,NAO APLICAVEL,2,MEDICO,4,1,ANO,30.0,DIA,4,12.0,0,DESCONHECIDO,NaN,3c04663c8c6a69bba2510ab19fa043d1d6670b81e42857...
1,BR-ANVISA-300208322,SP,EMPRESAS FARMACEUTICAS,EMPRESA FARMACEUTICA,BR-ANVISA-300208322,2023-09-01,2023-09-01,NaT,NOTIFICACAO ESPONTANEA,None,NaT,None,None,None,Feminino,Năo,Năo,None,None,Cefaleia,Năo,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,DESCONHECIDO,2021-01-22,NaT,...,1,NĂO,2,NĂO,2,NĂO,2,DESCONHECIDO,0,SUSPEITO,1,NAO APLICAVEL,2,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,2,0,DESCONHECIDO,NaN,DESCONHECIDO,0,NaN,0,DESCONHECIDO,NaN,716905c5a2cc460902e5682c94bd1379f674208b0c6ef5...


In [4]:
%%sql 
select * from fat_reacoes limit 2

Running query in 'duckdb:///:memory:'

,id,IDENTIFICACAO_NOTIFICACAO,HASH_BRONZE,HASH_SILVER,DATA_INICIO_HORA,DATA_FINAL_HORA,ATUALIZADO,LLT_CHAVE,GRAVE_CHAVE,DESFECHO_CHAVE,DURACAO_TIPO_CHAVE,DURACAO_VALOR,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA
0,BR-ANVISA-300000004,BR-ANVISA-300000004,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0b...,5e664ee6fcb8b1bf52f1a7aa7ef1853a166bf9cedfc2ff...,NaT,NaT,2025-11-17,4114,2,3,4,3.0,0,0,0,0,0,0
1,BR-ANVISA-300000005,BR-ANVISA-300000005,21a5c47062a6ba174af4058e70e681ff97ba91db760676...,497e70fa95ec2b543e9cb065388b857d545c25059f7f69...,2018-11-22,2018-11-22,2025-11-17,9107,1,3,0,NaN,0,0,0,0,1,0


In [6]:
%%sql 
select * from dim_soc_llt limit 2

Running query in 'duckdb:///:memory:'

,SOC_CHAVE,SOC_VALOR,HLGT,HLT,PT,LLT_CHAVE,REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual


# 5. Classificando atc_level_5 para estudo

In [12]:
%%sql
DROP TABLE IF EXISTS fat_analytics;

CREATE TABLE fat_analytics AS
with fat_med as (
SELECT 
    fat_medicamentos.IDENTIFICACAO_NOTIFICACAO, 
    atc.ATC_CODE_LEVEL_4_LEVEL_NAME,
    (
        CASE 
            WHEN atc.ATC_CODE_LEVEL_4 IN ('L04AB', 'L01FA', 'L04AC','L04AF') 
                THEN (
                    CASE 
                        /*Rituximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%rituximab%' THEN 'L01FA01_Rituximab'
                        /*Abatacept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%abatacept%' THEN 'L04AA24_Abatacept'
                        /*Etanercept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%etanercept%' THEN 'L04AB01_Etanercept'
                        /*Infliximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%infliximab%' THEN 'L04AB02_Infliximab'
                        /*Adalimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%adalimumab%' THEN 'L04AB04_Adalimumab'
                        /*Certolizumab Pegol*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab%' THEN 'L04AB05_Certolizumab Pegol'   
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumabe pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        
                        /*Golimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%golimumab%' THEN 'L04AB06_Golimumab'
                        /*Tocilizumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tocilizumab%' THEN 'L04AC07_Tocilizumab'
                        /*Baricitinib */
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%baricitinib%' THEN 'L04AF02_Baricitinib'
                        /*Upadacitinib*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%upadacitinib%' THEN 'L04AF03_Upadacitinib' 
                        ELSE 'missing'
                    END
                )
            ELSE 'Outros'
        END
    ) AS ATC_CODE_LEVEL_5_LEVEL_NAME
FROM fat_medicamentos
INNER JOIN dim_atc AS atc
    ON fat_medicamentos.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4 
    AND atc.ATC_LEVEL = 4
)
select 
dim.IDENTIFICACAO_NOTIFICACAO
,dim.PESO_KG
,fat_med.ATC_CODE_LEVEL_4_LEVEL_NAME AS ATC_LEVEL_4
,fat_med.ATC_CODE_LEVEL_5_LEVEL_NAME AS ATC_LEVEL_5  
,dim_soc_llt.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR AS REACAO
from dim_notificacoes as dim
inner join fat_med using (IDENTIFICACAO_NOTIFICACAO)
inner join fat_reacoes using (IDENTIFICACAO_NOTIFICACAO)
left join dim_soc_llt on fat_reacoes.LLT_CHAVE = dim_soc_llt.LLT_CHAVE

Running query in 'duckdb:///:memory:'

,Success


In [13]:
%%sql
select * from fat_analytics limit 2

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,PESO_KG,ATC_LEVEL_4,ATC_LEVEL_5,REACAO
0,BR-ANVISA-300342622,None,"CORTICOSTEROIDS, PLAIN",Outros,Reação alérgica
1,BR-ANVISA-300342622,None,CORTICOSTEROIDS,Outros,Reação alérgica


In [15]:
%%sql df <<
select * from fat_analytics

Running query in 'duckdb:///:memory:'

In [20]:
df.to_parquet("C:/Users/silma/Projetos/vigimed/data/03_gold/fat_analytical/fat_analytical.parquet", index=False)